In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_DIR))
print(PROJECT_DIR)

In [ ]:
import os
import random

import cv2
import librosa
import numpy as np
import matplotlib.pyplot as plt

from src.core.config import settings, update_path_settings
from src.domain.pipelines.annotations import (
    load_annotation_file,
    clean_annotation_dataframe,
    get_annotation_file,
)
from src.domain.pipelines.audio import (
    annotations_to_boxes,
    get_logical_windows,
    recalculate_annotations,
)
from src.domain.pipelines.image import (
    get_expected_spectrogram_shape,
    annotations_to_pixel_coords,
    get_crop_window,
    translate_boxes_to_yolo,
    audio_to_rgb_spectrogram,
    pad_to_min_size,
)
from src.domain.utils.species import get_species_id

update_path_settings(project_dir=PROJECT_DIR)

DATA_RAW_DIR = settings.DATA_RAW_DIR or Path("data/raw")
DATA_PREPROCESSED_DIR = settings.DATA_PREPROCESSED_DIR or Path("data/preprocessed")

In [ ]:
specie_dirs = [d for d in DATA_RAW_DIR.iterdir() if d.is_dir()]
sample_specie_dir = random.choice(specie_dirs)
sample_specie_dir

In [ ]:
records_files = list(DATA_RAW_DIR.glob("**/*.wav"))
sample_record = random.choice(records_files)

rel_path = sample_record.relative_to(DATA_RAW_DIR)
annotations_dir = (DATA_PREPROCESSED_DIR / rel_path).parent
sample_annotation = get_annotation_file(annotations_dir, sample_record)

print(f"Procesando audio: {sample_record}")

if sample_annotation:
    raw_df = load_annotation_file(sample_annotation)
    clean_df = clean_annotation_dataframe(raw_df)

    full_annotations = annotations_to_boxes(clean_df)
    print(f"Se encontraron {len(full_annotations)} eventos en el audio completo.")
else:
    print(
        "El archivo no tiene anotaciones. Ejecuta la celda de nuevo para buscar otro."
    )
    full_annotations = []

In [ ]:
SR = 32000
NFFT = 2048
HOP = 256
CLIP_DURATION = 5.0
CROP_SIZE = 640

audio_duration = librosa.get_duration(path=sample_record)
img_h, img_w = get_expected_spectrogram_shape(
    duration_sec=CLIP_DURATION, sample_rate=SR, n_fft=NFFT, hop_length=HOP
)


offsets = get_logical_windows(
    audio_duration, clip_duration_sec=CLIP_DURATION, overlap=0.1
)

target_offset = 0.0
clip_anns = []
for offset in offsets:
    clip_anns = recalculate_annotations(full_annotations, offset, CLIP_DURATION)
    if clip_anns:
        target_offset = offset
        break

In [ ]:
global_pixel_boxes = annotations_to_pixel_coords(
    annotations=clip_anns,
    duration=CLIP_DURATION,
    sample_rate=SR,
    img_w=img_w,
    img_h=img_h,
    class_mapping=lambda x, _: get_species_id(x),
)

ancla_pixel_box = global_pixel_boxes[0]

crop_x_start, crop_y_start = get_crop_window(
    target_event_bbox_px=ancla_pixel_box, img_w=img_w, img_h=img_h, crop_size=CROP_SIZE
)

yolo_labels = translate_boxes_to_yolo(
    global_bboxes_px=global_pixel_boxes,
    x_start=crop_x_start,
    y_start=crop_y_start,
    crop_size=CROP_SIZE,
)

In [ ]:
waveform, _ = librosa.load(
    sample_record, sr=SR, offset=target_offset, duration=CLIP_DURATION
)

espectrograma = audio_to_rgb_spectrogram(
    waveform=waveform, sample_rate=SR, n_fft=NFFT, hop_length=HOP
)

assert espectrograma.shape[:2] == (
    img_h,
    img_w,
), "Las dimensiones generadas no coinciden con la predicción matemática."

espectrograma_recortado = espectrograma[
    crop_y_start : crop_y_start + CROP_SIZE, crop_x_start : crop_x_start + CROP_SIZE
].copy()

imagen_final = pad_to_min_size(espectrograma_recortado, min_size=CROP_SIZE)

print(f"Forma de la imagen final procesada: {imagen_final.shape}")

In [ ]:
img_to_show = imagen_final.copy()

plt.figure(figsize=(10, 10))
for label in yolo_labels:
    class_id = label["class_id"]
    xc = label["xc_rel"] * CROP_SIZE
    yc = label["yc_rel"] * CROP_SIZE
    w = label["w_rel"] * CROP_SIZE
    h = label["h_rel"] * CROP_SIZE

    x1 = int(max(0, xc - w / 2))
    y1 = int(max(0, yc - h / 2))
    x2 = int(min(CROP_SIZE, xc + w / 2))
    y2 = int(min(CROP_SIZE, yc + h / 2))

    cv2.rectangle(img_to_show, (x1, y1), (x2, y2), (255, 0, 0), 2)
    plt.text(
        x1, max(0, y1 - 5), f"Clase {class_id}", color="red", fontsize=10, weight="bold"
    )

plt.imshow(img_to_show)
plt.axis("off")
plt.title(f"Parche 640x640 centrado en vocalización")
plt.show()